In [ ]:
#Import all libraries

import pandas as pd
import numpy as np
import re

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

from sklearn.model_selection import train_test_split

In [ ]:
#Load the Stanford Sentiment Treebank 2 dataset

data = load_dataset("glue","sst2")
df = pd.DataFrame(data['train'])


In [ ]:
#Viewing the dataset with labels

df = df[["sentence","label"]]
df.head()

In [ ]:
# X-> input sentences , y-> output labels

X = df['sentence']
y = df['label']

In [ ]:
X.shape

In [ ]:
# Tokenization

token = Tokenizer(num_words=10000)
token.fit_on_texts(X)

Xsequences = token.texts_to_sequences(X)

In [ ]:
# Padding

max = 40
Xpadded = pad_sequences(Xsequences, maxlen = max)

In [ ]:
# Train/Test Split

Xtrain, Xtest, ytrain, ytest = train_test_split(Xpadded,y, test_size=0.2,random_state=40)

In [ ]:
# RNN Model Building

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=32))
model.add(SimpleRNN(32, dropout=0.5))
model.add(Dense(1, activation="sigmoid"))

model.build(input_shape=(None, max))

In [ ]:
# Compile the model

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

model.summary()

In [ ]:
hist = model.fit(
    Xtrain,
    ytrain,
    epochs=5,
    batch_size=32,
    validation_data=(Xtest,ytest)
)

In [ ]:
# Calculate Validation Accuracy

loss,accuracy = model.evaluate(Xtest,ytest)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(hist.history['accuracy']); plt.plot(hist.history['val_accuracy'])
plt.title("Accuracy vs Epoch"); plt.show()

plt.plot(hist.history['loss']); plt.plot(hist.history['val_loss'])
plt.title("Loss vs Epoch"); plt.show()

In [ ]:
# Custom input prediction
s = input("Enter a sentence: ").lower()

seq = token.texts_to_sequences([s])

padded = pad_sequences(seq,maxlen=max)

prediction = model.predict(padded)

if prediction[0][0] > 0.5:
    print("Sentiment: Positive :)")
else:
    print("Sentiment: Negative :(")

In [ ]:
# LSTM Model Building, Compiling & Training

from tensorflow.keras.layers import LSTM

lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=10000, output_dim=32))
lstm_model.add(LSTM(32, dropout=0.5))
lstm_model.add(Dense(1, activation="sigmoid"))

lstm_model.build(input_shape=(None, max))

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

lstm_model.summary()

lstm_hist = lstm_model.fit(
    Xtrain,
    ytrain,
    epochs=5,
    batch_size=32,
    validation_data=(Xtest, ytest)
)

In [ ]:
# GRU Model Building, Compiling & Training

from tensorflow.keras.layers import GRU

gru_model = Sequential()
gru_model.add(Embedding(input_dim=10000, output_dim=32))
gru_model.add(GRU(32, dropout=0.5))
gru_model.add(Dense(1, activation="sigmoid"))

gru_model.build(input_shape=(None, max))

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

gru_model.summary()

gru_hist = gru_model.fit(
    Xtrain,
    ytrain,
    epochs=5,
    batch_size=32,
    validation_data=(Xtest, ytest)
)

In [ ]:
# Compare RNN vs LSTM vs GRU

import matplotlib.pyplot as plt

# Accuracy Comparison
plt.plot(hist.history['val_accuracy'], label='RNN')
plt.plot(lstm_hist.history['val_accuracy'], label='LSTM')
plt.plot(gru_hist.history['val_accuracy'], label='GRU')
plt.title("Validation Accuracy vs Epoch")
plt.legend(); plt.show()

# Loss Comparison
plt.plot(hist.history['val_loss'], label='RNN')
plt.plot(lstm_hist.history['val_loss'], label='LSTM')
plt.plot(gru_hist.history['val_loss'], label='GRU')
plt.title("Validation Loss vs Epoch")
plt.legend(); plt.show()